# Run the RCA Algorithm on All `03LIC_1071` PVLO Alarm Episodes

This notebook runs the third-party **Root Cause Analysis (RCA)** script
(`DATA/Scripts_updated_RCA_manish/Scripts/RCA.py`, developed by another team) on every
alarm episode under
`RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/all_episodes/`.

> **Nomenclature:** the folders are named `episode_NNNN`, but each one is actually a **cluster**.
> Folder `episode_NNNN` corresponds to `cluster_id == NNNN` in the workbook.

**How the RCA script expects its inputs**
- A **parquet** file with a `TimeStamp` column + one numeric column per tag (e.g. `03LIC_1071.PV`).
- An **alarm timestamp**. Internally it takes the `rca_duration_to_consider` (= 90) minutes
  of data *before* that timestamp for the pre-alarm analysis.
- `Config.json` (next to the script) for the target tag, dependent tags, algorithm params,
  the equipment knowledge-graph, and `paths.normal_data_path` → the no-alarm **benchmark**
  parquet used as the normal/baseline reference.

**What we do here (without changing the algorithm)**
- Each episode folder holds `episode_XXXX_pv_data.csv` (PV/OP data from 4 h before to 1 h after
  the alarm). Since the script reads *parquet*, we simply write each episode's PV data to a
  temporary parquet and call `RCA.run(parquet, alarm_timestamp)` — the RCA code itself is
  imported and used **unmodified**.
- The **alarm timestamp** for each folder is taken from the workbook's first sheet
  (`alarm_clusters`): we look up its `cluster_id` and use that cluster's **`cluster_start_time`**.
  This is the true onset of the episode's alarm (~4 h into the window), so the RCA gets its full
  90 minutes of pre-alarm history. (Using the first `AlarmStatus == 'ON'` minute instead would be
  wrong, because a *preceding* alarm can bleed into the window's left edge.)

Per-episode results (top-5 candidate root-cause tags) are collected into a table and saved to
`RESULTS/rca_algorithm_all_episodes/`.

## 1. Setup: import the (unmodified) RCA module and its Config.json

In [23]:
import sys, os, glob, time, tempfile, warnings, contextlib, io
from collections import Counter
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

# --- Paths ---
SCRIPTS_DIR  = '/home/h604827/ControlActions/DATA/Scripts_updated_RCA_manish/Scripts'
EPISODES_DIR = '/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/all_episodes'
ALARM_XLSX   = '/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx'
OUT_DIR      = '/home/h604827/ControlActions/RESULTS/rca_algorithm_all_episodes'
os.makedirs(OUT_DIR, exist_ok=True)

# --- Import the RCA script as a module (used unchanged) ---
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
import RCA

# Config.json that sits next to RCA.py (also resolves the benchmark/normal parquet)
CONFIG = RCA.load_config()
CFG    = RCA.get_data_path_configurations(CONFIG)
TARGET_TAG = CFG['analysis_settings']['target_tag']

print('RCA module     :', RCA.__file__)
print('Target tag     :', TARGET_TAG)
print('Pre-alarm window:', CFG['rca_duration_to_consider'], 'min before alarm')
print('# dependent tags:', len(CFG['dependent_tags']))
benchmark = RCA.get_normal_data_file_path(CONFIG)
print('Benchmark parquet:', benchmark, '(exists:', os.path.exists(benchmark), ')')

RCA module     : /home/h604827/ControlActions/DATA/Scripts_updated_RCA_manish/Scripts/RCA.py
Target tag     : 03LIC_1071.PV
Pre-alarm window: 90 min before alarm
# dependent tags: 24
Benchmark parquet: /home/h604827/ControlActions/DATA/Scripts_updated_RCA_manish/Scripts/benchmark_1071_no_alarms.parquet (exists: True )


## 2. Cluster lookup (workbook first sheet → each cluster's start/end)

Folder `episode_NNNN` == `cluster_id NNNN`. We read the `alarm_clusters` sheet and build a
per-cluster table so each folder can look up its own `cluster_start_time` (the alarm anchor)
and `cluster_end_time`.

In [24]:
# First sheet of the workbook = alarm_clusters. A folder named "episode_NNNN" actually
# denotes CLUSTER NNNN (nomenclature shift: the folders say "episode" but mean "cluster").
alarm_clusters = pd.read_excel(ALARM_XLSX, sheet_name=0)
for c in ['alarm_start', 'alarm_end', 'cluster_start_time', 'cluster_end_time']:
    if c in alarm_clusters.columns:
        alarm_clusters[c] = pd.to_datetime(alarm_clusters[c])
print('alarm_clusters sheet:', alarm_clusters.shape)

# One row per cluster_id: its start/end times and the alarm episodes it groups.
_grp = alarm_clusters.groupby('cluster_id')
cluster_info = pd.DataFrame({
    'cluster_start_time':  _grp['cluster_start_time'].first(),
    'cluster_end_time':    _grp['cluster_end_time'].first(),
    'n_alarms_in_cluster': _grp.size(),
    'episode_nums':        _grp['episode_num'].apply(list),
})
print('clusters:', len(cluster_info),
      '| cluster_id range', int(cluster_info.index.min()), '-', int(cluster_info.index.max()))

def get_cluster_info(cluster_id):
    """Return this cluster's start/end times and member alarm episodes, or {} if absent."""
    if cluster_id in cluster_info.index:
        r = cluster_info.loc[cluster_id]
        return dict(cluster_start_time=r['cluster_start_time'],
                    cluster_end_time=r['cluster_end_time'],
                    n_alarms_in_cluster=int(r['n_alarms_in_cluster']),
                    episode_nums=r['episode_nums'])
    return {}

alarm_clusters sheet: (1379, 19)
clusters: 539 | cluster_id range 1 - 539


## 3. Per-episode runner

For one folder: read its PV CSV → derive `cluster_id` from the folder name → look up that
cluster's `cluster_start_time` from the workbook as the alarm timestamp → write a temporary
parquet → call `RCA.run(...)` → parse the ranked root-cause tags. The RCA code is untouched;
we only feed it the data in the format it expects.

In [25]:
_TMP = os.path.join(tempfile.gettempdir(), 'rca_episode_tmp.parquet')

def run_rca_on_episode(ep_dir, verbose=False):
    ep = os.path.basename(ep_dir.rstrip('/'))
    cluster_id = int(ep.split('_')[1])            # folder episode_NNNN == cluster NNNN
    csv = os.path.join(ep_dir, f'{ep}_pv_data.csv')
    rec = {'episode_folder': ep, 'cluster_id': cluster_id, 'status': 'ok', 'note': '', 'error': ''}
    try:
        df = pd.read_csv(csv)
        df['TimeStamp'] = pd.to_datetime(df['TimeStamp'])
        df = df.sort_values('TimeStamp').reset_index(drop=True)
        rec['data_start'] = df['TimeStamp'].min()
        rec['data_end']   = df['TimeStamp'].max()
        rec['n_rows']     = len(df)

        # Alarm timestamp = this cluster's start time from the workbook's first sheet.
        # (NOT the first AlarmStatus=='ON' minute, which can be a preceding alarm that
        #  bleeds into the window's 4h-before edge and would leave ~1 pre-alarm row.)
        cinfo = get_cluster_info(cluster_id)
        rec['cluster_start_time']  = cinfo.get('cluster_start_time')
        rec['cluster_end_time']    = cinfo.get('cluster_end_time')
        rec['n_alarms_in_cluster'] = cinfo.get('n_alarms_in_cluster')
        rec['episode_nums']        = ', '.join(map(str, cinfo.get('episode_nums') or []))
        alarm_ts = cinfo.get('cluster_start_time')
        if alarm_ts is None or pd.isna(alarm_ts):
            raise ValueError(f'No cluster_start_time found for cluster_id {cluster_id}')
        rec['alarm_start_used'] = alarm_ts
        rec['pre_alarm_rows']   = int((df['TimeStamp'] <= alarm_ts).sum())

        df.to_parquet(_TMP, index=False)          # RCA expects parquet
        buf = io.StringIO()
        t0 = time.time()
        if verbose:
            top = RCA.run(_TMP, str(alarm_ts))
        else:
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                top = RCA.run(_TMP, str(alarm_ts))
        rec['elapsed_sec'] = round(time.time() - t0, 2)

        top = top or []
        rec['n_results'] = len(top)
        if not top:
            rec['status'] = 'no_result'
            reason = ''
            for line in buf.getvalue().splitlines():
                if any(k in line for k in ('Insufficient', 'failed', 'Error', 'Warning',
                                           'minimum', 'required', 'empty', 'No valid')):
                    reason = line.strip()
            rec['note'] = reason[:200]
        for k in range(5):
            rec[f'rank_{k+1}']     = top[k] if k < len(top) else ''
            rec[f'rank_{k+1}_tag'] = top[k].split(' - ')[0] if k < len(top) else ''
        rec['top_tags_full'] = ' | '.join(top)
    except Exception as e:
        rec['status'] = 'error'
        rec['error']  = str(e)[:300]
    return rec

## 4. Sanity check on a single episode (verbose)

Shows the raw RCA log for `episode_0001` so we can see the full pipeline (SSD → 8-test causal
analysis → ranked tags).

In [26]:
demo = run_rca_on_episode(os.path.join(EPISODES_DIR, 'episode_0027'), verbose=True)
print('\n--- parsed record ---')
for k in ['episode_folder', 'cluster_id', 'n_alarms_in_cluster', 'episode_nums',
          'cluster_start_time', 'cluster_end_time', 'alarm_start_used',
          'pre_alarm_rows', 'n_results', 'elapsed_sec', 'status']:
    print(f'  {k:19s}: {demo.get(k)}')
print('  top-5 root-cause tags:')
for k in range(1, 6):
    if demo.get(f'rank_{k}'):
        print(f'     {k}. {demo[f"rank_{k}"]}')

Root cause analysis execution started for 03LIC_1071.PV @ 2022-02-10 05:34:10.252000
Successfully loaded operating limits and PV data for 03LIC_1071.PV
Transient State execution started
Change points: CV=10, meanShift=9, slope=10, combined=19
Earliest transition indices found: 10 to 55
Change points: CV=11, meanShift=11, slope=6, combined=22
Earliest transition indices found: 40 to 75
Change points: CV=12, meanShift=13, slope=9, combined=22
Earliest transition indices found: 26 to 55
Change points: CV=11, meanShift=5, slope=9, combined=15
Earliest transition indices found: 6 to 89
Change points: CV=5, meanShift=6, slope=11, combined=18
Earliest transition indices found: 9 to 64
Change points: CV=8, meanShift=6, slope=6, combined=14
Earliest transition indices found: 25 to 49
Change points: CV=16, meanShift=0, slope=12, combined=21
Earliest transition indices found: 12 to 89
Change points: CV=14, meanShift=10, slope=14, combined=18
Earliest transition indices found: 64 to 89
Change poin

## 5. Run the RCA on every episode

Sequential loop over all folders (~1 s/episode). Set `MAX_EPISODES` to a small number for a
quick trial run.

In [27]:
MAX_EPISODES = None   # e.g. 20 for a quick trial; None = all

episode_dirs = sorted(glob.glob(os.path.join(EPISODES_DIR, 'episode_*')))
if MAX_EPISODES:
    episode_dirs = episode_dirs[:MAX_EPISODES]
print(f'Running RCA on {len(episode_dirs)} episode(s)...\n')

results = []
t_start = time.time()
for i, ep_dir in enumerate(episode_dirs, 1):
    results.append(run_rca_on_episode(ep_dir, verbose=False))
    if i % 25 == 0 or i == len(episode_dirs):
        ok = sum(r['status'] == 'ok' for r in results)
        nr = sum(r['status'] == 'no_result' for r in results)
        er = sum(r['status'] == 'error' for r in results)
        print(f'  [{i:4d}/{len(episode_dirs)}]  ok={ok}  no_result={nr}  error={er}  '
              f'elapsed={time.time()-t_start:5.0f}s')
print(f'\nDone in {time.time()-t_start:.0f}s.')

Running RCA on 539 episode(s)...

  [  25/539]  ok=25  no_result=0  error=0  elapsed=   51s
  [  50/539]  ok=50  no_result=0  error=0  elapsed=   97s
  [  75/539]  ok=75  no_result=0  error=0  elapsed=  128s
  [ 100/539]  ok=100  no_result=0  error=0  elapsed=  156s
  [ 125/539]  ok=125  no_result=0  error=0  elapsed=  186s
  [ 150/539]  ok=150  no_result=0  error=0  elapsed=  216s
  [ 175/539]  ok=175  no_result=0  error=0  elapsed=  244s
  [ 200/539]  ok=200  no_result=0  error=0  elapsed=  273s
  [ 225/539]  ok=225  no_result=0  error=0  elapsed=  302s
  [ 250/539]  ok=250  no_result=0  error=0  elapsed=  331s
  [ 275/539]  ok=275  no_result=0  error=0  elapsed=  359s
  [ 300/539]  ok=300  no_result=0  error=0  elapsed=  387s
  [ 325/539]  ok=325  no_result=0  error=0  elapsed=  415s
  [ 350/539]  ok=350  no_result=0  error=0  elapsed=  444s
  [ 375/539]  ok=375  no_result=0  error=0  elapsed=  472s
  [ 400/539]  ok=400  no_result=0  error=0  elapsed=  500s
  [ 425/539]  ok=425  no_

## 6. Collect, save, and review results

In [28]:
res_df = pd.DataFrame(results)

front = ['episode_folder', 'cluster_id', 'n_alarms_in_cluster', 'episode_nums',
         'alarm_start_used', 'cluster_start_time', 'cluster_end_time',
         'data_start', 'data_end', 'n_rows', 'pre_alarm_rows',
         'status', 'n_results', 'elapsed_sec',
         'rank_1', 'rank_2', 'rank_3', 'rank_4', 'rank_5',
         'rank_1_tag', 'rank_2_tag', 'rank_3_tag', 'rank_4_tag', 'rank_5_tag',
         'top_tags_full', 'note', 'error']
cols = [c for c in front if c in res_df.columns] + [c for c in res_df.columns if c not in front]
res_df = res_df[cols]

csv_out  = os.path.join(OUT_DIR, 'rca_results_all_episodes.csv')
xlsx_out = os.path.join(OUT_DIR, 'rca_results_all_episodes.xlsx')
res_df.to_csv(csv_out, index=False)
res_df.to_excel(xlsx_out, index=False)

# Also refresh the copy that lives next to the 1071 workbook (read by the merge-back step below)
episodes_root_xlsx = os.path.normpath(os.path.join(EPISODES_DIR, '..', 'rca_results_all_episodes.xlsx'))
res_df.to_excel(episodes_root_xlsx, index=False)

print('Saved:')
print('  ', csv_out)
print('  ', xlsx_out)
print('  ', episodes_root_xlsx)
print('\nStatus counts:')
print(res_df['status'].value_counts().to_string())

Saved:
   /home/h604827/ControlActions/RESULTS/rca_algorithm_all_episodes/rca_results_all_episodes.csv
   /home/h604827/ControlActions/RESULTS/rca_algorithm_all_episodes/rca_results_all_episodes.xlsx
   /home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/rca_results_all_episodes.xlsx

Status counts:
status
ok           537
no_result      2


In [29]:
# Per-episode (= per-cluster) top-5 root-cause tags
view_cols = ['episode_folder', 'cluster_id', 'alarm_start_used', 'pre_alarm_rows', 'status',
             'rank_1_tag', 'rank_2_tag', 'rank_3_tag', 'rank_4_tag', 'rank_5_tag']
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 200)
res_df[view_cols]

,episode_folder,cluster_id,alarm_start_used,pre_alarm_rows,status,rank_1_tag,rank_2_tag,rank_3_tag,rank_4_tag,rank_5_tag
0,episode_0001,1,2022-01-05 08:53:41.853,240,ok,03PI_1814.PV,03PIC_1068.PV,02FI_1000.PV,03PIC_3131.PV,03PI_1495.PV
1,episode_0002,2,2022-01-07 09:55:16.253,240,ok,03PI_1495.PV,03PIC_1068.PV,03LIC_1085.PV,02FI_1000.PV,03LIC_3178.PV
2,episode_0003,3,2022-01-07 13:33:25.702,240,ok,03PI_1495.PV,03PIC_1068.PV,03PIC_3131.PV,03PI_1814.PV,03PIC_1104.PV
3,episode_0004,4,2022-01-07 14:17:11.555,240,ok,03TI_1901.PV,03PI_1495.PV,03LIC_1085.PV,03TI_1081.PV,03PI_1814.PV
4,episode_0005,5,2022-01-07 14:54:23.554,240,ok,03TIC_1092.PV,02FI_1000.PV,03PI_1495.PV,03PI_1814.PV,03TI_1421.PV
...,...,...,...,...,...,...,...,...,...,...
534,episode_0535,535,2025-06-21 22:15:17.359,240,ok,03TIC_1092.PV,03TIC_1145.PV,03PIC_1068.PV,03LIC_1085.PV,03LIC_1097.PV
535,episode_0536,536,2025-06-22 13:13:26.264,240,ok,03FIC_3415.PV,03LIC_1085.PV,03LIC_1097.PV,03TI_1081.PV,03PIC_1104.PV
536,episode_0537,537,2025-06-22 14:32:47.231,240,ok,03PIC_1013.PV,03PIC_1104.PV,03PIC_1068.PV,03PIC_3131.PV,03LIC_1085.PV
537,episode_0538,538,2025-06-22 15:55:07.321,240,ok,03PI_1814.PV,03TIC_1092.PV,03LIC_1085.PV,03LIC_1097.PV,03PIC_1068.PV


In [ ]:
# Aggregate: how often each tag is picked as the #1 root cause, and across the top-5
ok_df = res_df[res_df['status'] == 'ok']
print(f'Episodes with results : {len(ok_df)} / {len(res_df)}')

print('\nMost frequent RANK-1 root-cause tags:')
print(ok_df['rank_1_tag'].value_counts().head(15).to_string())

counter = Counter()
for _, r in ok_df.iterrows():
    for k in range(1, 6):
        t = r.get(f'rank_{k}_tag', '')
        if t:
            counter[t] += 1
freq = (pd.DataFrame(counter.most_common(), columns=['tag', 'appearances_in_top5'])
        if counter else pd.DataFrame(columns=['tag', 'appearances_in_top5']))
freq.to_csv(os.path.join(OUT_DIR, 'rca_tag_frequency.csv'), index=False)
print('\nMost frequent tags across TOP-5 (all episodes):')
print(freq.head(15).to_string(index=False))

Episodes with results : 537 / 539

Most frequent RANK-1 root-cause tags:
rank_1_tag
03LIC_1085.PV    137
03PI_1495.PV      88
03TIC_1145.PV     62
03TIC_1092.PV     38
03LIC_3178.PV     31
03LIC_1097.PV     31
03PIC_1068.PV     23
02FI_1000.PV      21
03PIC_1104.PV     20
03FIC_3415.PV     16
03PI_1814.PV      16
03PIC_1013.PV     13
03TI_1421.PV      12
03TI_1015.PV      10
03TI_1081.PV       6

Most frequent tags across TOP-5 (all episodes):
          tag  appearances_in_top5
03LIC_1085.PV                  346
 03PI_1495.PV                  281
03LIC_1097.PV                  231
03PIC_1068.PV                  210
03TIC_1145.PV                  173
03TIC_1092.PV                  168
03PIC_1104.PV                  157
 02FI_1000.PV                  135
03LIC_3178.PV                  132
03PIC_3131.PV                  126
 03TI_1081.PV                  115
 03TI_1421.PV                   98
03FIC_3415.PV                   98
 03PI_1814.PV                   91
 03TI_1015.PV              

In [31]:
# Episodes that produced no result (usually too little pre-alarm data), with the reason
nores = res_df[res_df['status'] != 'ok'][['episode_folder', 'alarm_start_used',
                                           'pre_alarm_rows', 'status', 'note', 'error']]
print(f'{len(nores)} episode(s) without a top-5 result:')
nores

2 episode(s) without a top-5 result:


,episode_folder,alarm_start_used,pre_alarm_rows,status,note,error
488,episode_0489,2025-04-17 19:15:34.002,0,no_result,ValueError: PV data for 03LIC_1071.PV has only...,
522,episode_0523,2025-05-31 21:13:19.260,20,no_result,Warning: No combined scores accumulated.,


## 7. Add the RCA rank columns back into the 1071 workbook

Merge `rank_1_tag` … `rank_5_tag` (keyed by `cluster_id`) into **both** sheets of
`03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx`:

- **`alarm_clusters`** (episode level): every episode row of a cluster receives that cluster's
  RCA tags, so a cluster that merges several alarm episodes repeats the same tags on each row.
- **`control_actions`**: every control-action row receives its cluster's RCA tags.

Only the five `rank_*_tag` columns are added. A one-time backup of the original workbook is
written next to it before it is updated, and the step is re-runnable (existing rank columns are
replaced, not duplicated).

In [32]:
import shutil

RANK_COLS = ['rank_1_tag', 'rank_2_tag', 'rank_3_tag', 'rank_4_tag', 'rank_5_tag']

# The 1071 workbook to update, and the RCA results (both now live in the episodes folder)
EPISODES_ROOT = '/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219'
WORKBOOK = os.path.join(EPISODES_ROOT, '03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx')
RCA_XLSX = os.path.join(EPISODES_ROOT, 'rca_results_all_episodes.xlsx')

# RCA rank columns, one row per cluster_id
rca_ranks = pd.read_excel(RCA_XLSX)[['cluster_id'] + RANK_COLS].copy()
rca_ranks[RANK_COLS] = rca_ranks[RANK_COLS].fillna('')
print('RCA rank rows:', len(rca_ranks), '| clusters:', rca_ranks['cluster_id'].nunique())

# One-time safety backup of the original workbook (delete later if not needed)
backup = WORKBOOK.replace('.xlsx', '_backup_pre_rca.xlsx')
if not os.path.exists(backup):
    shutil.copy2(WORKBOOK, backup)
    print('Backup created :', os.path.basename(backup))
else:
    print('Backup exists  :', os.path.basename(backup))

# Merge the rank columns into every sheet that has cluster_id (idempotent: drop old rank cols first)
sheets = pd.read_excel(WORKBOOK, sheet_name=None)   # dict of all sheets, order preserved
for name, df in sheets.items():
    df = df.drop(columns=[c for c in RANK_COLS if c in df.columns], errors='ignore')
    if 'cluster_id' in df.columns:
        df = df.merge(rca_ranks, on='cluster_id', how='left')
        df[RANK_COLS] = df[RANK_COLS].fillna('')
        n_with_tags = int((df['rank_1_tag'] != '').sum())
        print(f"  {name:16s}: {df.shape[0]:6d} rows | rank cols added ({n_with_tags} rows have RCA tags)")
    else:
        print(f"  {name:16s}: no cluster_id -> left unchanged")
    sheets[name] = df

# Write all sheets back to the same workbook
with pd.ExcelWriter(WORKBOOK, engine='openpyxl') as xw:
    for name, df in sheets.items():
        df.to_excel(xw, sheet_name=name, index=False)
print('\nUpdated workbook saved:', WORKBOOK)

RCA rank rows: 539 | clusters: 539
Backup exists  : 03LIC_1071_pvlo_alarms_clustered_with_control_actions_backup_pre_rca.xlsx
  alarm_clusters  :   1379 rows | rank cols added (1348 rows have RCA tags)
  control_actions :  42290 rows | rank cols added (41869 rows have RCA tags)

Updated workbook saved: /home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx


In [33]:
# Verify the update: both sheets now carry the rank columns; spot-check one multi-row cluster
chk = pd.read_excel(WORKBOOK, sheet_name=None)
for name, df in chk.items():
    print(f"{name}: {df.shape} | rank cols = {[c for c in df.columns if c.startswith('rank_')]}")

ac = chk['alarm_clusters']
demo_cid = 27
print(f"\nalarm_clusters rows for cluster_id={demo_cid} (same tags repeated per episode):")
print(ac.loc[ac['cluster_id'] == demo_cid, ['episode_num', 'cluster_id'] + RANK_COLS].to_string(index=False))

ca = chk['control_actions']
print(f"\ncontrol_actions sample for cluster_id={demo_cid}:")
print(ca.loc[ca['cluster_id'] == demo_cid, ['cluster_id', 'Source'] + RANK_COLS].head(4).to_string(index=False))

alarm_clusters: (1379, 19) | rank cols = ['rank_1_tag', 'rank_2_tag', 'rank_3_tag', 'rank_4_tag', 'rank_5_tag']
control_actions: (42290, 17) | rank cols = ['rank_1_tag', 'rank_2_tag', 'rank_3_tag', 'rank_4_tag', 'rank_5_tag']

alarm_clusters rows for cluster_id=27 (same tags repeated per episode):
 episode_num  cluster_id    rank_1_tag    rank_2_tag    rank_3_tag    rank_4_tag   rank_5_tag
          34          27 03LIC_1085.PV 03PIC_1068.PV 03PIC_1104.PV 03PIC_3131.PV 02FI_1000.PV

control_actions sample for cluster_id=27:
 cluster_id     Source    rank_1_tag    rank_2_tag    rank_3_tag    rank_4_tag   rank_5_tag
         27 03LIC_1034 03LIC_1085.PV 03PIC_1068.PV 03PIC_1104.PV 03PIC_3131.PV 02FI_1000.PV
         27 03LIC_1034 03LIC_1085.PV 03PIC_1068.PV 03PIC_1104.PV 03PIC_3131.PV 02FI_1000.PV
         27 03LIC_1034 03LIC_1085.PV 03PIC_1068.PV 03PIC_1104.PV 03PIC_3131.PV 02FI_1000.PV
         27 03LIC_1034 03LIC_1085.PV 03PIC_1068.PV 03PIC_1104.PV 03PIC_3131.PV 02FI_1000.PV
